# Lab 2: Model Optimization Techniques - Part A: Quantization

Welcome to lab 2 of the TinyML course! In this lab, we are going to learn a few approaches for deep learning model optimization. These techniques, when combined, can significantly shrink model size and make them suitable for model deployment. The techniques we are going to cover today are:

* Quantization
* Model Pruning
* Knowledge Distillation

As this is the first lab in our course that utilizes Jupyter Notebook, let's begin by setting up our environment and installing all required dependencies.

## Installing Required Packages

For this lab, we need both `tensorflow` and `tensorflow-model-optimization`. In addition, we also need `numpy`, `tf_keras` and `tqdm`. We use `tensorflow` to build and train a neural network, and `tensorflow-model-optimization` for the model optimization experiments we are going to run. Please note that as of now, **Python 3.14 is not yet supported**, and we recommend you to use a Python version between 3.9 and 3.13. You can use a standalone virtual environment or Conda environment to run a different version from your system installation.

In [ ]:
# Uninstall packages on Google Colab because new versions of TensorFlow has changed some TFLite APIs
#%pip uninstall -y tensorflow tensorflow-model-optimization

# a) Install with CPU only support:
%pip install tensorflow<2.21
# b) Alternatively, with CUDA (GPU) support on Linux:
#%pip install tensorflow[with-cuda]<2.21

%pip install numpy tensorflow-model-optimization tf_keras tqdm

### Note: Resolving `keras.src` Namespace Issue

When using TensorFlow and TensorFlow Model Optimization in Colab, you may encounter a `keras.src` namespace issue, causing incompatibility with `tensorflow_model_optimization.quantization.keras`. To resolve this:

1. Set the `KERAS_BACKEND` environment variable to `tensorflow` before importing TensorFlow.
2. Ensure you are using compatible versions of TensorFlow (`>=2.12`) and TensorFlow Model Optimization.
3. Clone the model using `tensorflow.keras.models.clone_model()` to ensure it aligns with the `tensorflow.keras` namespace.
4. Always restart the runtime and reinstall TensorFlow-related packages to avoid lingering conflicts.

This ensures that all operations use the correct `tensorflow.keras` implementation, avoiding compatibility issues.

In [ ]:
import os

# Apply workaround before imports
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

## Part I: Post Training Quantization (PTQ) - Integer, Dynamic Range, and Float16 Data Types
First part of this notebook demonstrates three types of post-training quantization for a CNN model trained on the MNIST dataset. Quantization is a model compression technique that reduces model size and computational requirements, enabling efficient deployment on resource-constrained devices.

### Goals
1. Train a CNN model on the MNIST dataset.
2. Apply three quantization techniques:
   - Integer Quantization
   - Dynamic Range Quantization
   - Float 16 Quantization
3. Compare the quantized models in terms of size and accuracy.

### Dataset Preparation
We use the MNIST dataset, which contains grayscale images of handwritten digits (0-9).
1. Normalize the pixel values to the range [0, 1].
2. Reshape the data for input into the CNN model.
3. One-hot encode the labels for classification.

In [ ]:
# Import required libraries
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

# Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalize the data
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Reshape the data for CNN input
x_train = x_train.reshape(-1, 28, 28, 1)
x_test = x_test.reshape(-1, 28, 28, 1)

# One-hot encode the labels
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"Training data shape: {x_train.shape}, Labels shape: {y_train.shape}")
print(f"Test data shape: {x_test.shape}, Labels shape: {y_test.shape}")


### Training a Simple CNN

We build a Convolutional Neural Network (CNN) with the following layers:
1. **Convolutional Layer**: Extracts features from the input images.
2. **MaxPooling Layer**: Reduces spatial dimensions, lowering computational requirements.
3. **Flatten Layer**: Converts the 2D feature maps into a 1D vector.
4. **Dense Layers**: Fully connected layers for classification.

The model is compiled using the Adam optimizer and trained for 2 epochs.

In [ ]:
from tensorflow.data import Dataset
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

# Build a simple CNN model
def create_cnn_model():
    return Sequential([
        Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
        MaxPooling2D((2, 2)),
        Flatten(),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

# Create training and validation datasets
train_dataset = Dataset.from_tensor_slices((x_train, y_train))
val_dataset = Dataset.from_tensor_slices((x_test, y_test))
batch_size = 32

# Compile and train the model
model = create_cnn_model()
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_dataset.batch(batch_size), epochs=2, validation_data=val_dataset.batch(batch_size))

# Get validation accuracy
original_accuracy = history.history["val_accuracy"][-1]

### Applying Quantization Techniques

We apply three types of post-training quantization to the trained CNN model:
1. **Integer Quantization**: Converts both weights and activations to 8-bit integers.
2. **Dynamic Range Quantization**: Reduces the precision of weights while keeping activations in floating-point format.
3. **Float 16 Quantization**: Converts both weights and activations to 16-bit floating-point values.

In [ ]:
import tensorflow as tf
from tensorflow import lite as tf_lite

# ------------------------------
# BASELINE: No quantization
# ------------------------------

# Convert the original Keras model to TFLite format without any quantization
converter = tf_lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

# Save the baseline model to a .tflite file
with open("model_baseline.tflite", "wb") as f:
    f.write(tflite_model)

# ------------------------------
# FULL INTEGER QUANTIZATION
# ------------------------------

# Define a representative generator required for calibrating quantization
# Note: input data must match model input shape and dtype (float32)
def representative_data_gen(representative_size=256):
    for x_batch, _ in val_dataset.take(representative_size).batch(1):
        yield [x_batch]

# Create the converter from the Keras model again
converter = tf_lite.TFLiteConverter.from_keras_model(model)

# Enable default optimizations (includes quantization)
converter.optimizations = [tf_lite.Optimize.DEFAULT]

# Provide representative dataset for calibration
converter.representative_dataset = representative_data_gen

# Specify that only int8 operations are to be used in the model
converter.target_spec.supported_ops = [tf_lite.OpsSet.TFLITE_BUILTINS_INT8]

# Set input and output tensor types to int8
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

# Convert and save the INT8 quantized model
tflite_model_int8 = converter.convert()
with open("model_integer_quant.tflite", "wb") as f:
    f.write(tflite_model_int8)

# ------------------------------
# DYNAMIC RANGE QUANTIZATION
# ------------------------------

# This quantization method does not require a representative dataset
# Only weights are quantized; activations stay in float
converter = tf_lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf_lite.Optimize.DEFAULT]

# Convert and save the dynamic range quantized model
tflite_model_dynamic = converter.convert()
with open("model_dynamic_quant.tflite", "wb") as f:
    f.write(tflite_model_dynamic)

# ------------------------------
# FLOAT16 QUANTIZATION
# ------------------------------

# Convert the model to use float16 weights (useful for GPUs or TPUs that support float16)
converter = tf_lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf_lite.Optimize.DEFAULT]

# Set the supported types to float16
converter.target_spec.supported_types = [tf.float16]

# Convert and save the float16 quantized model
tflite_model_float16 = converter.convert()
with open("model_float16_quant.tflite", "wb") as f:
    f.write(tflite_model_float16)


### Comparison of Model Sizes and Accuracy

We compare the quantized models in terms of:
1. **Model Size**: Smaller models are better suited for resource-constrained devices.
2. **Accuracy**: Quantization should preserve the original model’s accuracy as much as possible.

In [ ]:
import os

import numpy as np
from tqdm import tqdm

# Compare model sizes
model_files = [
    "model_baseline.tflite",
    "model_integer_quant.tflite",
    "model_dynamic_quant.tflite",
    "model_float16_quant.tflite"
]

print("Model Sizes (KB):")
for file in model_files:
    print(f"{file}: {os.path.getsize(file) / 1024:.2f} KB")

# Evaluate accuracy of a TFLite model on the test set
def evaluate_tflite_model(tflite_model_path, batch_size=32):
    # Load the TFLite model
    interpreter = tf_lite.Interpreter(model_path=tflite_model_path)

    # Get input and output tensor details
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    # Get input information for evaluation
    input_info = input_details[0]
    input_dtype = input_info['dtype']
    input_scale, input_zero_point = input_info['quantization']

    # Change batch size for inference
    interpreter.resize_tensor_input(input_info['index'], [batch_size, 28, 28, 1])

    # Allocate space for tensors
    interpreter.allocate_tensors()

    correct_predictions = 0
    n_batch = 0

    for x_batch, y_batch_onehot in tqdm(val_dataset.batch(batch_size, drop_remainder=True)):
        input_data = x_batch

        # Quantize the input if model expects int8
        if input_dtype == np.int8:
            input_data = input_data / input_scale + input_zero_point
            input_data = np.round(input_data).astype(np.int8)
        elif input_dtype == np.float16:
            input_data = input_data.astype(np.float16)

        # Set the tensor and invoke the interpreter
        interpreter.set_tensor(input_info['index'], input_data)
        interpreter.invoke()

        # Get predictions
        output_data = interpreter.get_tensor(output_details[0]['index'])
        preds = output_data.argmax(1)
        y_batch = y_batch_onehot.numpy().argmax(1)
        # Calculate and accumulate number of correct predictions
        correct_predictions += (preds == y_batch).sum().item()
        n_batch += 1

    # Return classification accuracy
    return correct_predictions / (n_batch * batch_size)

# Display accuracy of each model
print("\nModel Accuracies:")
print(f"Baseline: {original_accuracy:.4f}\n")
print(f"\nInteger Quantization: {evaluate_tflite_model('model_integer_quant.tflite'):.4f}\n")
print(f"\nDynamic Range Quantization: {evaluate_tflite_model('model_dynamic_quant.tflite'):.4f}\n")
print(f"\nFloat 16 Quantization: {evaluate_tflite_model('model_float16_quant.tflite'):.4f}")


### Part I Summary

Quantization significantly reduces model size while maintaining comparable accuracy. Key observations:
- Integer Quantization provides the smallest model size and computation complexity but may slightly reduce accuracy.
- Dynamic Range Quantization balances size, computation complexity, and performance.
- Float 16 Quantization retains higher accuracy with moderate size and computation complexity reduction.


## Part II: Quantization-Aware Training (QAT) - Integer and Dynamic Range
Quantization-Aware Training is a technique where integer quantization is simulated during the training process. This allows the model to adjust its weights and activations, minimizing the accuracy loss caused by quantization.

In this step, we:
1. Prepare a quantization-aware model using TensorFlow’s `QuantizeWrapper`.
2. Train the model on MNIST with quantization simulation.
3. Convert the model to TensorFlow Lite format for Integer and Dynamic Range.
4. Compare model sizes and accuracy after QAT.


In [ ]:

from tensorflow_model_optimization.quantization.keras import quantize_model

# Prepare the quantization-aware model
qat_model = quantize_model(create_cnn_model())

# Compile the QAT model
qat_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Train the QAT model
print("Training the QAT Model...")
qat_history = qat_model.fit(train_dataset.batch(batch_size), epochs=2, validation_data=val_dataset.batch(batch_size))

# Get validation accuracy of the QAT model
qat_accuracy = qat_history.history["val_accuracy"][-1]

### Applying Quantization After QAT

We now convert the QAT-trained model to TensorFlow Lite format and apply three quantization techniques:
1. **Integer Quantization**
2. **Dynamic Range Quantization**

---

### 🧪 Lab 2 Submission Reminder

**Please complete the code below and take a screenshot of it as part of your Lab 2 submission.**

⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️⬇️

---


In [ ]:
# Convert the QAT-trained Baseline model to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
tflite_model = converter.convert()
with open("qat_model_baseline.tflite", "wb") as f:
    f.write(tflite_model)

# Integer Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_types = [tf.int8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model_qat_int8 = converter.convert()

# Save Integer Quantization model
with open("qat_model_integer_quant.tflite", "wb") as f:
    f.write(tflite_model_qat_int8)

# Dynamic Range Quantization
converter = tf.lite.TFLiteConverter.from_keras_model(qat_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_qat_dynamic = converter.convert()

# Save Dynamic Range Quantization model
with open("qat_model_dynamic_quant.tflite", "wb") as f:
    f.write(tflite_model_qat_dynamic)


### Comparison of Model Sizes and Accuracy

We compare the model sizes and accuracy for the quantized QAT-trained models.

In [ ]:
# Compare model sizes
model_files = [
    "qat_model_baseline.tflite",
    "qat_model_integer_quant.tflite",
    "qat_model_dynamic_quant.tflite",
]

print("\nModel Sizes After QAT (KB):")
for file in model_files:
    print(f"{file}: {os.path.getsize(file) / 1024:.2f} KB")

# Evaluate accuracy for QAT-quantized models
print("\nModel Accuracies After QAT:")
print(f"QAT Baseline Model Accuracy: {qat_accuracy:.4f}\n")
print(f"\nInteger Quantization: {evaluate_tflite_model('qat_model_integer_quant.tflite'):.4f}\n")
print(f"\nDynamic Range Quantization: {evaluate_tflite_model('qat_model_dynamic_quant.tflite'):.4f}")


######################################################################################
# Please take a screenshot of the following result as part of your Lab 2 submission. #
######################################################################################


---
⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️⬆️

### 📸 Lab 2 Submission Reminder

**Please take a screenshot of the above result and include it as part of your Lab 2 submission.**

---


### Part II Summary

Quantization-Aware Training (QAT) significantly reduces accuracy loss caused by quantization. Key observations:
- QAT improves the accuracy of quantized models, making it suitable for resource-constrained deployments that demands the high precision/accuracy.
- QAT is incompatible with Float16 quantization because they target fundamentally different quantization formats.